# Simulación meteorológica

## Identificación del equipo

| Nombre completo | No. de cuenta |
|---|---|
| Jonathan Hernández Lazcano | 200417 |
| Camila Rodriguez Rosas | 194100 |


## Fase 1. Particionamiento

Se requiere la segmentación del espacio de soluciones (el espacio de simulación), donde una región meteorológica puede estar definida por un espacio bidimensional *NxN* dividido en *P* franjas horizontales de aproximadamente *N/P* filas. Cada proceso almacena su franja en una matriz local (temperatura / presión atmosférica), optimizando la memoria caché.

Cuando *N* no es divisible entre *P*, las primeras `N % P` franjas reciben una fila extra.

In [1]:
import numpy as np

# Filas asignadas a cada proceso y fila global donde inicia su franja
def particion_filas(rank, n_proc, N):
  base, extra = divmod(N, n_proc)
  subgrid_rows = base + (1 if rank < extra else 0)
  offset = rank * base + min(rank, extra)
  return subgrid_rows, offset

# Asignación de cada proceso en una franja del dominio global
def inicializacion_subdominio(rank, n_proc, N, pert=1.01):
  subgrid_rows, offset = particion_filas(rank, n_proc, N)
  cols = N
  # Creación de una matriz local agregando bordes extras (ghost cells)
  grid = np.zeros((subgrid_rows+2, cols+2), dtype=np.float64)
  # Configuración inicial: perturbación (tormenta) centrada en el dominio global
  centro = N / 2
  gr = offset + np.arange(subgrid_rows)[:, None]   # filas globales de la franja
  c = np.arange(cols)[None, :]
  dist = np.sqrt((gr - centro)**2 + (c - centro)**2)
  grid[1:subgrid_rows+1, 1:cols+1] = 100 * pert * np.exp(-dist/120)
  return grid, subgrid_rows, cols

## Fase 2. Comunicación

En esta fase, el estado de cada una de las celdas tiene un límite del subdominio asignado de manera local y se analiza a través de iteraciones *t+1*; al mismo tiempo, se requiere conocer las lecturas de los vecinos. Para evitar colisiones (acceso simultáneo a la misma memoria) se agregan filas adicionales (*ghost cells*). De esta manera, antes de llevar a cabo los cálculos los vecinos intercambian sus bordes.

En los bordes globales (sin vecino) se aplica una condición física de frontera de flujo cero: la ghost cell copia la fila interior adyacente.

In [2]:
from multiprocessing import Pipe

# Intercambiar bordes con los vecinos Norte y Sur, y aplicar condiciones físicas en la frontera global
def intercambio_ghost_cells(grid, subgrid_rows, pipe_top, pipe_bottom):
  # Intercambio hacia el norte
  if pipe_top is not None:
    pipe_top.send(grid[1,:])
    grid[0,:] = pipe_top.recv()
  # Intercambio hacia el sur
  if pipe_bottom is not None:
    pipe_bottom.send(grid[subgrid_rows,:])
    grid[subgrid_rows+1,:] = pipe_bottom.recv()
  # Modificación de condiciones físicas (solo en bordes sin vecino)
  if pipe_top is None:
    grid[0,:] = grid[1,:]
  if pipe_bottom is None:
    grid[subgrid_rows+1,:] = grid[subgrid_rows,:]

## Fase 3. Agrupamiento y replicación

En esta fase, con las *ghost cells* ya actualizadas, cada proceso ejecuta la simulación física de su franja de manera independiente y en paralelo (stencil de 5 puntos).

In [3]:
# Actualización de las temperaturas/presión atmosférica
def calcular_paso_atmosferico(grid, subgrid_rows, cols):
  grid_next = grid.copy()
  # Aplicar modificación de puntos sobre operaciones vectorizadas
  grid_next[1:subgrid_rows+1, 1:cols+1] = 0.25 * (
      grid[0:subgrid_rows, 1:cols+1] +   # Modificación de condiciones del vecino del Norte
      grid[2:subgrid_rows+2, 1:cols+1] + # Modificación de condiciones del vecino del Sur
      grid[1:subgrid_rows+1, 0:cols] +   # Modificación de condiciones del vecino del Oeste
      grid[1:subgrid_rows+1, 2:cols+2]   # Modificación de condiciones del vecino del Este
  )
  return grid_next

## Fase 4. Integración

En esta fase se coordinan los trabajos de los diferentes *Workers* (procesos), permitiendo la medición del tiempo de cómputo y de comunicación y coordinando los canales de comunicación `pipes`.

In [4]:
import time

# Ciclo principal SPMD que orquesta comunicación y cómputo
def worker_spmd(rank, num_proc, N, steps, pipe_top, pipe_bottom, result_pipe, perturbacion=1.01):
  grid, subgrid_rows, cols = inicializacion_subdominio(rank, num_proc, N, perturbacion)
  historico = []
  t_comm = t_comp = 0.0
  muestreo = steps//10 if steps > 10 else 1
  for step in range(steps):
    # Comunicación de bordes
    t0 = time.perf_counter()
    intercambio_ghost_cells(grid, subgrid_rows, pipe_top, pipe_bottom)
    t1 = time.perf_counter()
    # Cómputo numérico
    grid = calcular_paso_atmosferico(grid, subgrid_rows, cols)
    t2 = time.perf_counter()
    t_comm += t1 - t0
    t_comp += t2 - t1
    # Muestreo periódico para visualización del fenómeno atmosférico
    if step % muestreo == 0:
      historico.append((step, grid[1:subgrid_rows+1, 1:cols+1].copy()))
  # Enviar el registro histórico, la franja final y los tiempos medidos
  result_pipe.send({
      "rank": rank,
      "final": grid[1:subgrid_rows+1, 1:cols+1].copy(),
      "historico": historico,
      "t_comm": t_comm,
      "t_comp": t_comp,
  })
  result_pipe.close()

## Fase 5. Evaluación de desempeño

`ejecutar_simulacion` crea un `Pipe` por cada par de franjas vecinas y uno de resultados por proceso, lanza los *P* procesos y mide el **tiempo de pared** desde el arranque hasta el `join` (incluye creación de procesos, comunicación y cómputo). Después reensambla la malla global.

> Se usa el contexto `fork`: en macOS/Windows el método por defecto (`spawn`) no puede serializar funciones definidas dentro de una libreta de Jupyter. En Linux/Colab `fork` ya es el valor por defecto.

In [5]:
import multiprocessing as mp

ctx = mp.get_context("fork")

def ejecutar_simulacion(P, N, steps, perturbacion=1.01):
  # Pipes entre vecinos: vecinos[i] conecta la franja i (abajo) con la franja i+1 (arriba)
  vecinos = [ctx.Pipe() for _ in range(P-1)]
  resultados = [ctx.Pipe(duplex=False) for _ in range(P)]
  procesos = []
  t0 = time.perf_counter()
  for rank in range(P):
    pipe_top = vecinos[rank-1][1] if rank > 0 else None
    pipe_bottom = vecinos[rank][0] if rank < P-1 else None
    p = ctx.Process(target=worker_spmd,
                    args=(rank, P, N, steps, pipe_top, pipe_bottom, resultados[rank][1], perturbacion))
    p.start()
    procesos.append(p)
  datos = [r[0].recv() for r in resultados]
  for p in procesos:
    p.join()
  tp = time.perf_counter() - t0
  datos.sort(key=lambda d: d["rank"])
  malla = np.vstack([d["final"] for d in datos])
  return {
      "Tp": tp,
      "malla": malla,
      "historico": [d["historico"] for d in datos],
      "t_comm": max(d["t_comm"] for d in datos),
      "t_comp": max(d["t_comp"] for d in datos),
  }

# Prueba rápida
r = ejecutar_simulacion(4, 100, 50)
print(r["malla"].shape, f"Tp = {r['Tp']:.3f} s")

(100, 100) Tp = 0.011 s


### Experimento

Se ejecuta la simulación con **P ∈ {1, 2, 4, 8}** para **N = 500** (tabla principal) y **N = 100** (comparación para la pregunta 1), con 1000 pasos de tiempo. Cada configuración se repite 3 veces y se reporta la mediana. También se verifica que la malla final en paralelo sea idéntica a la secuencial.

In [6]:
import os
import pandas as pd

PROCESOS = [1, 2, 4, 8]
STEPS = 1000
REPETICIONES = 3
print("Núcleos disponibles:", os.cpu_count())

def benchmark(N):
  filas = []
  referencia = None
  for P in PROCESOS:
    corridas = [ejecutar_simulacion(P, N, STEPS) for _ in range(REPETICIONES)]
    tp = float(np.median([c["Tp"] for c in corridas]))
    t_comm = float(np.median([c["t_comm"] for c in corridas]))
    t_comp = float(np.median([c["t_comp"] for c in corridas]))
    malla = corridas[0]["malla"]
    if referencia is None:
      referencia = malla
    assert np.allclose(malla, referencia), f"La malla con P={P} no coincide con la secuencial"
    filas.append({"P": P, "Tp": tp, "t_comm": t_comm, "t_comp": t_comp})
  df = pd.DataFrame(filas)
  T1 = df.loc[0, "Tp"]
  df["Sp"] = T1 / df["Tp"]
  df["Ep"] = df["Sp"] / df["P"]
  df["% comunicación"] = 100 * df["t_comm"] / (df["t_comm"] + df["t_comp"])
  return df, referencia

df500, malla500 = benchmark(500)
df100, malla100 = benchmark(100)
print("Mallas paralelas idénticas a la secuencial: OK")

Núcleos disponibles: 10


Mallas paralelas idénticas a la secuencial: OK


### Tabla de evaluación (N = 500, 1000 pasos)

In [7]:
def tabla_evaluacion(df):
  return pd.DataFrame({
      "Procesos (P)": ["1 (Secuencial)" if p == 1 else str(p) for p in df["P"]],
      "Tiempo Total TP (s)": df["Tp"].map("{:.3f}".format),
      "Aceleración (SP=T1/TP)": df["Sp"].map("{:.2f}".format),
      "Eficiencia (EP=SP/P)": df["Ep"].map("{:.0%}".format),
  })

tabla_evaluacion(df500)

,Procesos (P),Tiempo Total TP (s),Aceleración (SP=T1/TP),Eficiencia (EP=SP/P)
0,1 (Secuencial),0.372,1.00,100%
1,2,0.222,1.68,84%
2,4,0.158,2.35,59%
3,8,0.177,2.11,26%


### Comparación con N = 100

In [8]:
tabla_evaluacion(df100)

,Procesos (P),Tiempo Total TP (s),Aceleración (SP=T1/TP),Eficiencia (EP=SP/P)
0,1 (Secuencial),0.022,1.00,100%
1,2,0.029,0.76,38%
2,4,0.041,0.53,13%
3,8,0.077,0.29,4%


### Desglose de comunicación vs. cómputo (proceso más lento)

In [9]:
desglose = pd.concat([df100.assign(N=100), df500.assign(N=500)])[
    ["N", "P", "Tp", "t_comm", "t_comp", "% comunicación"]]
desglose.round(4)

,N,P,Tp,t_comm,t_comp,% comunicación
0,100,1,0.0220,0.0004,0.0169,2.3925
1,100,2,0.0289,0.0120,0.0109,52.4413
2,100,4,0.0412,0.0286,0.0070,80.2908
3,100,8,0.0772,0.0637,0.0064,90.8687
0,500,1,0.3721,0.0007,0.3554,0.1845
1,500,2,0.2220,0.0160,0.1864,7.9057
2,500,4,0.1580,0.0315,0.1041,23.2186
3,500,8,0.1767,0.0994,0.0656,60.2421


### Gráficas

In [10]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for df, N in [(df100, 100), (df500, 500)]:
  ax[0].plot(df["P"], df["Tp"], "o-", label=f"N={N}")
  ax[1].plot(df["P"], df["Sp"], "o-", label=f"N={N}")
  ax[2].plot(df["P"], 100*df["Ep"], "o-", label=f"N={N}")
ax[1].plot(PROCESOS, PROCESOS, "k--", label="Ideal")
ax[1].axhline(10, color="r", ls=":", label="Límite Amdahl (f=10%)")
ax[0].set(title="Tiempo total", xlabel="P", ylabel="Tp (s)")
ax[1].set(title="Aceleración", xlabel="P", ylabel="Sp")
ax[2].set(title="Eficiencia", xlabel="P", ylabel="Ep (%)")
for a in ax:
  a.set_xticks(PROCESOS)
  a.grid(alpha=.3)
  a.legend()
plt.tight_layout()
plt.show()

Matplotlib is building the font cache; this may take a moment.


/var/folders/5k/bspdkwtd22s4q3_x8gvdhmx40000gn/T/ipykernel_934/1856599947.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
r = ejecutar_simulacion(4, 500, STEPS)
pasos = [s for s, _ in r["historico"][0]]
idx = [0, len(pasos)//2, len(pasos)-1]
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a, i in zip(ax, idx):
  campo = np.vstack([h[i][1] for h in r["historico"]])
  im = a.imshow(campo, cmap="inferno")
  a.set_title(f"Paso {pasos[i]} (P=4, N=500)")
  for k in range(1, 4):   # fronteras entre franjas
    a.axhline(sum(particion_filas(j, 4, 500)[0] for j in range(k)) - .5, color="c", lw=.8, ls="--")
fig.colorbar(im, ax=ax, shrink=.8)
plt.show()

/var/folders/5k/bspdkwtd22s4q3_x8gvdhmx40000gn/T/ipykernel_934/468626675.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Preguntas

In [12]:
def fila(df, P):
  return df[df["P"] == P].iloc[0]

for N, df in [(100, df100), (500, df500)]:
  f8 = fila(df, 8)
  print(f"N={N}: % comunicación con P=8 = {f8['% comunicación']:.1f}%,  Sp(8) = {f8['Sp']:.2f},  Ep(8) = {f8['Ep']:.0%}")

bytes_franja = (500 + 2) * 8
print(f"Ghost cell por mensaje con N=500: {bytes_franja} B; por paso y proceso interior: {4*bytes_franja} B (2 envíos + 2 recepciones)")

N=100: % comunicación con P=8 = 90.9%,  Sp(8) = 0.29,  Ep(8) = 4%
N=500: % comunicación con P=8 = 60.2%,  Sp(8) = 2.11,  Ep(8) = 26%
Ghost cell por mensaje con N=500: 4016 B; por paso y proceso interior: 16064 B (2 envíos + 2 recepciones)


### 1. ¿Cómo afecta el tamaño de la franja de Ghost Cells enviada por los pipes al tiempo total de ejecución cuando se incrementa la resolución de la malla a N=500?

Cada intercambio envía una fila completa de `N + 2` valores `float64`. Con N=500 son 502 × 8 B ≈ **4 KB por mensaje**, y un proceso interior manda dos y recibe dos en cada paso. Ese volumen crece de forma **lineal, O(N)**, mientras que el cómputo de cada franja crece como **O(N²/P)**. Por eso, al pasar de N=100 a N=500 la comunicación aumenta unas 5 veces y el cómputo unas 25 veces, y la **relación comunicación/cómputo (superficie/volumen) disminuye**.

En la práctica, con N=100 el tiempo se va casi todo en serializar los arreglos con `pickle`, en las llamadas al sistema de los `pipes` y en la sincronización con los vecinos, así que agregar procesos casi no ayuda e incluso puede hacer el programa más lento. Con N=500 el costo de las *ghost cells* sigue presente, porque cada paso exige esperar al vecino y cruzar a kernel, pero queda amortizado por el trabajo numérico, y la aceleración y la eficiencia mejoran. La latencia por mensaje pesa más que el tamaño: 4 KB caben en el búfer del pipe, de modo que el costo dominante es el **número de sincronizaciones por paso**, no los bytes. La tabla de desglose muestra el % de tiempo en comunicación para ambos tamaños.

Para mallas más grandes, el tiempo total se puede reducir con franjas fantasma más anchas, que permiten intercambiar cada *k* pasos, con mensajes asíncronos que solapan comunicación y cómputo, o con memoria compartida en lugar de `pipes`.

### 2. Si una tormenta severa genera mayor carga de cálculo numérico en un subdominio específico de la cuadrícula, ¿por qué la asignación estática de tareas deja de ser eficiente y qué alternativa de balanceo dinámico de carga debería implementarse?

La partición estática reparte el mismo número de filas a cada proceso suponiendo que **todas las celdas cuestan lo mismo**. Si la tormenta concentra en una franja más cálculo (física más fina, más iteraciones del solver, refinamiento local), ese proceso se vuelve el **más lento**. Como el algoritmo es SPMD con sincronización en cada paso por el intercambio de *ghost cells*, **todos esperan al más lento**: el tiempo de cada paso es el máximo de los tiempos locales, no el promedio. Los demás procesadores se quedan ociosos y la eficiencia cae aunque el trabajo total sea el mismo.

Alternativas de balanceo dinámico:
- **Descomposición más fina con bolsa de tareas (*master–worker / work pool*):** dividir el dominio en muchos bloques pequeños, más que procesos, y repartirlos bajo demanda, por ejemplo con `multiprocessing.Pool.imap_unordered` y un `chunksize` pequeño. El proceso que termina toma el siguiente bloque, así que la zona de la tormenta se reparte sola entre varios trabajadores.
- ***Work stealing*:** cada proceso tiene su cola de bloques y, cuando se vacía, toma trabajo de la cola de un vecino cargado.
- **Reparticionamiento adaptativo:** medir el tiempo de cómputo por franja cada cierto número de pasos y mover las fronteras (o usar particiones ponderadas como curvas de Hilbert o ORB) para que cada proceso reciba aproximadamente el mismo **costo**, no el mismo número de filas.

En esta simulación, la opción más natural es el **reparticionamiento adaptativo por costo medido**, porque conserva la localidad de las *ghost cells*. Se puede combinar con bloques más finos en la región de la tormenta.

In [13]:
f = 0.10
S_max = 1 / f
for P in [2, 4, 8, 16, 64, 1024]:
  print(f"P={P:5d}: S_Amdahl = {1/(f + (1-f)/P):.2f}")
print(f"Límite P→∞: S_max = 1/f = {S_max:.0f}x")
print(f"Medido (N=500, P=8): Sp = {fila(df500, 8)['Sp']:.2f}  vs. Amdahl P=8: {1/(f + (1-f)/8):.2f}")

P=    2: S_Amdahl = 1.82
P=    4: S_Amdahl = 3.08
P=    8: S_Amdahl = 4.71
P=   16: S_Amdahl = 6.40
P=   64: S_Amdahl = 8.77
P= 1024: S_Amdahl = 9.91
Límite P→∞: S_max = 1/f = 10x
Medido (N=500, P=8): Sp = 2.11  vs. Amdahl P=8: 4.71


### 3. Si la fase de comunicación e intercambio de bordes representa un 10% del tiempo total inalterable en paralelo, ¿cuál es la aceleración máxima teórica alcanzable según la Ley de Amdahl sin importar el número de procesadores añadidos?

Ley de Amdahl, con fracción no paralelizable *f* = 0.10:

$$S(P) = \frac{1}{f + \frac{1-f}{P}} \quad\Longrightarrow\quad S_{max} = \lim_{P\to\infty} S(P) = \frac{1}{f} = \frac{1}{0.10} = \mathbf{10}$$

Sin importar cuántos procesadores se agreguen, la aceleración nunca superará **10×**, y la eficiencia tiende a 0 cuando *P* crece. Con P=8 el máximo teórico es 1 / (0.1 + 0.9/8) ≈ **4.71×**. La aceleración medida queda por debajo de ese valor porque, además del 10 %, el programa tiene costos que Amdahl no contempla: creación de procesos, serialización y espera entre vecinos.